# ♟️ ChessMarro Training Pipeline

This notebook provides a complete training pipeline for the ChessMarro chess AI model. All cells have been converted to functions, and there's an interactive configuration form at the end to select your dataset, model name, and base model.

## Usage:
1. Run all cells from top to bottom
2. Scroll to the bottom and run the last cell for the interactive training form
3. Choose your dataset, model name, and optionally load a base model to continue training

## Key Features:
- ✅ Modular function-based architecture
- 📊 Support for parquet datasets
- 🔧 Checkpoint recovery (resume from last epoch)
- 💾 Automatic model saving
- 🧪 Model quality testing

In [1]:
import random
import os
from torch.utils.data import Dataset, DataLoader
import chess
import chess.svg
from IPython.display import display, SVG
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import pyarrow as pa 
import pyarrow.parquet as pq
import torch
import numpy as np
import math

## 1️⃣ Setup & Imports
All necessary modules and custom chess library imports are set up below.

In [2]:
def read_data(file,page,size):
    with pq.ParquetFile(file) as pf:   
        iterb = pf.iter_batches(batch_size = size)
        for i in range(page):
            next(iterb)
        
        batches = next(iterb)

        df_chess = pa.Table.from_batches([batches]).to_pandas()
        
        batches = None
        iterb = None
        
        df_chess['board'] = df_chess['board'].apply(lambda board: board.reshape(77, 8, 8).astype(int))
        df_chess.info(memory_usage='deep')
        df_chess.memory_usage(deep=True)
        return df_chess


def get_total_pages(file_path, size):
    with pq.ParquetFile(file_path) as pf:
        total_rows = pf.metadata.num_rows
        total_pages = math.ceil(total_rows / size)
        return total_pages

## 2️⃣ Data Loading Functions
Functions to read and process chess datasets from parquet files.

In [3]:
def test_reading(file_path='lc0_converted.parquet_v2_suffled.gzip', page=0, size=300):
    df_chess = read_data(file_path, page, size)
    board = chess.Board(df_chess.loc[100, 'fen_original'])

    # Mostrar el tablero inicial
    svg_board = chess.svg.board(board=board, size=300)
    display(SVG(svg_board))
    print(df_chess.loc[100])
    print(df_chess.loc[100, 'fen_original'])

In [4]:
def create_dataLoaders(df_chess):    
    # This class is used by pytorch for providing data to model. We use it to read dataset and convert to desired format 
    class ChessDataset(Dataset):
        def __init__(self, df):
            self.dataframe = df

        def __len__(self):
            return len(self.dataframe)
        
        def __getitem__(self, idx):
            fen = torch.tensor(self.dataframe.loc[idx, 'board'], dtype=torch.float32)
            uci_best_move = self.dataframe.loc[idx, 'best']
            value = self.dataframe.loc[idx, 'value']
            return (fen, uci_best_move, value, self.dataframe.loc[idx, 'fen_original'])    

    # We can select a small sample to try the training algorithm
    # In this case we select all the Dataset provided
    df_selected = df_chess.sample(n=len(df_chess), random_state=42).reset_index(drop=True)

    # Then we create an Object with this dataset
    data_train = ChessDataset(df_selected)

    # Select 80% for training and 20% for testing
    train_size = int(0.8 * len(data_train))   
    test_size = len(data_train) - train_size
    print(len(data_train))
    train_dataset, test_dataset = torch.utils.data.random_split(data_train, [train_size, test_size])

    # Create the Dataloader 
    dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
    dataloader_test = DataLoader(test_dataset, batch_size=64, shuffle=True, drop_last=True)
    return (dataloader, dataloader_test)
    
def test_dataloader(file_path='lc0_converted.parquet.gzip', page=0, size=300):
    df_chess = read_data(file_path, page, size)
    (dataloader, dataloader_test) = create_dataLoaders(df_chess)
    for batch in dataloader:
        print("\nBest Move Matrix:")
        print(batch[1])
        break  # Imprimir the first to show only converted data

## 3️⃣ Model Architecture
Neural network definition with residual connections for chess position evaluation.

In [5]:
class Mish(nn.Module):
    """Activación Mish: x * tanh(softplus(x))"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block para atención de canales"""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlock(nn.Module):
    """Bloque Residual con Pre-activación y SE"""
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.se = SEBlock(channels)
        self.activation = Mish()

    def forward(self, x):
        residual = x
        out = self.bn1(x)
        out = self.activation(out)
        out = self.conv1(out)
        
        out = self.bn2(out)
        out = self.activation(out)
        out = self.conv2(out)
        
        out = self.se(out)
        return out + residual

class ChessNetPV_Optimized(nn.Module):
    def __init__(self, num_blocks=12): # 6 o 12 bloques es mucho más profundo que el original
        super(ChessNetPV_Optimized, self).__init__()

        in_channels = 77
        base_channels = 256 # Ancho constante profesional 
        head_bottleneck_channels = 32 # Para reducir parámetros en FC [6]
        

        # Entrada inicial
        self.conv_input = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1, bias=False)
        
        # Torre Residual (Cuerpo de la red)
        self.res_tower = nn.Sequential(
            *[ResBlock(base_channels) for _ in range(num_blocks)]
        )
        
        # BN y Activación final de la torre (por arquitectura de pre-activación)
        self.final_bn = nn.BatchNorm2d(base_channels)
        self.final_act = Mish()

        # --- CABEZAL DE POLÍTICA (4096 salidas) ---
        self.policy_conv = nn.Conv2d(base_channels, head_bottleneck_channels, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(head_bottleneck_channels)
        self.policy_fc = nn.Linear(head_bottleneck_channels * 8 * 8, 4096)

        # --- CABEZAL DE VALOR (1 salida) ---
        self.value_conv = nn.Conv2d(base_channels, head_bottleneck_channels, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(head_bottleneck_channels)
        self.value_fc1 = nn.Linear(head_bottleneck_channels * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        # Cuerpo
        x = self.conv_input(x)
        x = self.res_tower(x)
        x = self.final_act(self.final_bn(x))

        # Política
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = self.policy_fc(p)

        # Valor
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        value = F.relu(self.value_fc1(v))
        value = self.tanh(self.value_fc2(value))

        return policy, value

In [6]:
def initialize_model(base_model_path=None):
    """Initialize the model and device"""
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using {device} device")
    model = ChessNetPV_Optimized().to(device)
    
    # Load base model if provided
    if base_model_path and os.path.exists(base_model_path):
        model.load_state_dict(torch.load(base_model_path, map_location=device))
        print(f"✅ Loaded base model from {base_model_path}")
    
    return device, model

## 4️⃣ Model Training
Training functions with early stopping, loss calculation, and checkpoint management.

In [7]:
def train(model, dataloader, dataloader_test, device):
    max_epoch = 10 
    log_interval = 300
    early_stopping_patience = 5 
    best_loss = float('inf')
    best_accuracy = 0
    patience = 0 

    learning_rate = 0.005 
    # Criterios para las dos cabezas
    #criterion_policy = nn.CrossEntropyLoss()
    criterion_policy = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_value = nn.MSELoss() 
    
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  

    for epoch in range(max_epoch):
        # --- FASE DE ENTRENAMIENTO ---
        model.train()  
        pbar = tqdm(total=len(dataloader), desc=f'Training Epoch {epoch}')
        total_train_loss = 0  

        # Nota: El dataloader ahora debe devolver (data, target_policy, target_value)
        for batch_idx, (data, target_p, target_v, fen_original) in enumerate(dataloader):
            data = data.to(device)
            target_p = target_p.to(device) # El movimiento (0-4095)
            target_v = target_v.to(device).float() # El valor de lc0 (-1 a 1)

            optimizer.zero_grad()
            
            # La red ahora devuelve dos salidas
            policy_out, value_out = model(data)
            
            # Calculamos las pérdidas por separado
            loss_p = criterion_policy(policy_out, target_p)
            loss_v = criterion_value(value_out.squeeze(), target_v) # squeeze para ajustar dimensiones
            
            # Pérdida total combinada
            # Podemos dar pesos: p.ej. total_loss = loss_p + (1.0 * loss_v)
            loss = loss_p + (2 * loss_v)
            
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            if batch_idx % log_interval == 0:
                pbar.set_postfix(L_pol=loss_p.item(), L_val=loss_v.item())
            pbar.update(1)

        avg_train_loss = total_train_loss / len(dataloader)
        pbar.close()

        # --- FASE DE TEST ---
        model.eval()
        test_loss = 0
        correct_policy = 0
        total_mse_v = 0

        with torch.no_grad():
            for data, target_p, target_v, fen_original in dataloader_test:
                data = data.to(device)
                target_p = target_p.to(device)
                target_v = target_v.to(device).float()

                policy_out, value_out = model(data)
                
                # Pérdida en test
                l_p = criterion_policy(policy_out, target_p)
                l_v = criterion_value(value_out.squeeze(), target_v)
                test_loss += (l_p + l_v).item()
                
                # Métrica de precisión para la política (movimientos)
                pred = policy_out.argmax(dim=1, keepdim=True) 
                correct_policy += pred.eq(target_p.view_as(pred)).sum().item()
                
                # Métrica de error para el valor (opcional pero recomendado)
                total_mse_v += l_v.item()

        test_loss /= len(dataloader_test)
        accuracy = 100. * correct_policy / len(dataloader_test.dataset)
        avg_v_error = total_mse_v / len(dataloader_test)

        print(f'🔎 Test Total Loss: {test_loss:.4f} | Acc Policy: {accuracy:.2f}% | Value MSE: {avg_v_error:.4f}')

        # Early stopping basado en la pérdida total de test
        if test_loss < best_loss:
            best_loss = test_loss
            best_accuracy = accuracy
            patience = 0 
            # Aquí podrías guardar el modelo: torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience += 1
            if patience > early_stopping_patience:
                print('⚠️ Stopping early due to no improvement.')
                break

        scheduler.step() 

    print(f'🎯 Entrenamiento finalizado. Mejor precisión de movimiento: {best_accuracy:.2f}%')

In [8]:
def start_training_loop(model, device, dataset_file, model_name, start_epoch=0, num_epochs=7):
    """
    Training loop that iterates through dataset pages and trains the model
    
    Args:
        model: The neural network model
        device: torch device (cuda, mps, or cpu)
        dataset_file: Path to the parquet dataset file
        model_name: Name prefix for saved model checkpoints (without extension)
        start_epoch: Starting epoch number (default 0)
        num_epochs: Total number of epochs to train (default 7)
    """
    # --- Lógica de Recuperación ---
    start_iteration = start_epoch
    # Buscamos si existen modelos guardados para continuar desde el último
    checkpoints = [f for f in os.listdir() if f.startswith(model_name) and f.endswith('.pth')]
    
    if checkpoints:
        # Extraemos los números de las épocas y buscamos el máximo
        epochs = [int(f.replace(model_name, '').replace('_epoch', '').replace('.pth', '')) for f in checkpoints]
        if epochs:
            last_epoch = max(epochs)
            model_path = f'{model_name}_epoch{last_epoch}.pth'
            
            model.load_state_dict(torch.load(model_path, map_location=device))
            start_iteration = last_epoch + 1
            print(f"🚀 Checkpoint detectado: Cargado {model_path}. Reanudando en iteración {start_iteration}")

    if not checkpoints:
        print("🆕 No se han encontrado checkpoints. Empezando entrenamiento desde cero.")

    # --- Bucle de entrenamiento modificado ---
    for i in range(start_iteration, num_epochs):
        print(f"🔄 Training iteration {i}")
    
        # Shuffle dataset to ensure variety
        df_train = read_data(dataset_file, i, 200000)
        df_train = df_train.sample(frac=1, random_state=i)  # Shuffle
    
        # Create dataloaders
        (dataloader, dataloader_test) = create_dataLoaders(df_train)
    
        # Train model
        train(model, dataloader, dataloader_test, device)
    
        # Save model after each iteration
        torch.save(model.state_dict(), f'{model_name}_epoch{i}.pth')
        print(f"💾 Model checkpoint saved: {model_name}_epoch{i}.pth")

In [9]:
import random

def test_model_quality(model, file_path, device, n_samples=100):
    model.eval()
    
    # 1. Cargar el dataset y extraer muestra aleatoria
    print(f"Leendo datos de {file_path}...")
    df = read_data(file_path,1,2000)
    #df = pd.read_parquet(file_path)
    df_sample = df.sample(n=n_samples).reset_index(drop=True)
    
    top1_hits = 0
    top5_hits = 0
    
    print(f"Iniciando test sobre {n_samples} posiciones...")
    
    with torch.no_grad():
        for i, row in df_sample.iterrows():
            # Preprocesar la entrada (usa tu función actual)
            # Suponiendo que recibe el FEN y devuelve el tensor [1, planes, 8, 8]
            input_tensor = torch.tensor(row['board'], dtype=torch.float32).unsqueeze(0).to(device)
            
            # Obtener el índice de la jugada real (target)
            # Asegúrate de que 'move_index' es el nombre de tu columna de etiquetas
            target_idx = row['best'] 
            
            # Predicción
            policy_logits, value = model(input_tensor)
            
            # Calcular Top-K
            # Obtenemos los índices de las 5 mayores probabilidades
            _, top5_indices = torch.topk(policy_logits, 5, dim=1)
            top5_indices = top5_indices.squeeze().tolist() # Lista de 5 índices
            
            # Comprobación Top-1
            if top5_indices[0] == target_idx:
                top1_hits += 1
            
            # Comprobación Top-5
            if target_idx in top5_indices:
                top5_hits += 1
                
    # Resultados finales
    print("\n" + "="*30)
    print(f"📊 RESULTADOS DEL TEST (n={n_samples})")
    print("="*30)
    print(f"✅ Top-1 Accuracy: {top1_hits}%  (Jugada exacta)")
    print(f"🎯 Top-5 Accuracy: {top5_hits}%  (En el radar)")
    print(f"📉 Error promedio Value: {0.005} (Referencia)") # Opcional calcular el real
    print("="*30)

# Para llamarlo:
#test_model_quality(model, 'lc0_converted.parquet.gzip', device)

## 5️⃣ Model Testing & Evaluation
Functions to evaluate model performance on unseen data with Top-1 and Top-5 accuracy metrics.

## 6️⃣ Interactive Training Form
Run the cell below to start an interactive configuration form where you can:
- Choose your dataset
- Set the trained model name
- Load a base model (optional, for transfer learning)
- Configure training parameters

In [ ]:
# ============================================================================
# 🎯 MAIN EXECUTION - Interactive Configuration Form
# ============================================================================

def get_available_files(extension='.pth'):
    """Get list of available files with given extension in current directory"""
    return [f for f in os.listdir() if f.endswith(extension)]

def print_config_form():
    """Display the training configuration form with available options"""
    print("\n" + "="*70)
    print("🎮 CHESS TRAINING CONFIGURATION FORM")
    print("="*70)
    
    # Dataset options
    print("\n📊 AVAILABLE DATASETS:")
    datasets = get_available_files('.parquet.gzip') + get_available_files('.parquet')
    if datasets:
        for idx, ds in enumerate(datasets, 1):
            print(f"  [{idx}] {ds}")
    else:
        print("  ❌ No parquet files found. Using 'lc0_converted.parquet.gzip'")
        datasets = ['lc0_converted.parquet.gzip']
    
    # Model name input
    print("\n💾 MODEL NAME:")
    print("  Enter a name for your trained model (without extension)")
    print("  Example: 'chessmarro_v2', 'mymodel', etc.")
    
    # Base model options
    print("\n🔧 BASE MODEL (Optional):")
    base_models = get_available_files('.pth')
    if base_models:
        print("  [0] None (Train from scratch)")
        for idx, bm in enumerate(base_models, 1):
            print(f"  [{idx}] {bm}")
    else:
        print("  [0] None (Train from scratch)")
        print("  ❌ No .pth files found")
    
    print("\n" + "="*70)

def main_training_form():
    """Interactive form to configure and start training"""
    
    print_config_form()
    
    # Dataset selection
    datasets = get_available_files('.parquet.gzip') + get_available_files('.parquet')
    if not datasets:
        datasets = ['lc0_converted.parquet.gzip']
    
    print("\nSelect dataset [1]: ", end="")
    dataset_choice = input().strip() or "1"
    try:
        dataset_idx = int(dataset_choice) - 1
        dataset_file = datasets[dataset_idx] if 0 <= dataset_idx < len(datasets) else datasets[0]
    except:
        dataset_file = datasets[0]
    
    # Model name input
    print("\nEnter model name [chessmarro]: ", end="")
    model_name = input().strip() or "chessmarro"
    
    # Base model selection
    base_models = get_available_files('.pth')
    base_model_path = None
    
    if base_models:
        print("\nSelect base model [0]: ", end="")
        base_choice = input().strip() or "0"
        try:
            base_idx = int(base_choice) - 1
            if base_idx >= 0 and base_idx < len(base_models):
                base_model_path = base_models[base_idx]
        except:
            pass
    
    # Additional parameters
    max_epochs = get_total_pages(dataset_file, 200000)
    print("\nNumber of epochs ["+str(max_epochs)+"]: ", end="")
    num_epochs_input = input().strip() or str(max_epochs)
    try:
        num_epochs = int(num_epochs_input)
    except:
        num_epochs = 7
    
    print("\nStart epoch [0]: ", end="")
    start_epoch_input = input().strip() or "0"
    try:
        start_epoch = int(start_epoch_input)
    except:
        start_epoch = 0
    
    # Confirmation
    print("\n" + "="*70)
    print("⚙️  TRAINING CONFIGURATION:")
    print(f"  📊 Dataset: {dataset_file}")
    print(f"  💾 Model Name: {model_name}")
    print(f"  🔧 Base Model: {base_model_path if base_model_path else 'None (train from scratch)'}")
    print(f"  📈 Epochs: {num_epochs}")
    print(f"  🔄 Start Epoch: {start_epoch}")
    print("="*70)
    print("\nProceed with training? (yes/no) [yes]: ", end="")
    proceed = input().strip().lower() or "yes"
    
    if proceed in ['yes', 'y']:
        print("\n🚀 Starting training...\n")
        
        # Initialize model
        device, model = initialize_model(base_model_path)
        
        # Start training loop
        start_training_loop(model, device, dataset_file, model_name, start_epoch, num_epochs)
        
        # Save final model
        final_model_path = f'{model_name}_final.pth'
        torch.save(model.state_dict(), final_model_path)
        print(f"\n✅ Training completed! Final model saved to: {final_model_path}")
        
        # Optional: Test the model
        print("\nTest trained model? (yes/no) [no]: ", end="")
        test_choice = input().strip().lower() or "no"
        if test_choice in ['yes', 'y']:
            print("\nTesting model quality...")
            test_model_quality(model, dataset_file, device, n_samples=100)
        
        return model, device, model_name
    else:
        print("❌ Training cancelled.")
        return None, None, None

# Run the main form
if __name__ == "__main__":
    model, device, model_name = main_training_form()


🎮 CHESS TRAINING CONFIGURATION FORM

📊 AVAILABLE DATASETS:
  [1] lc0_converted.parquet.gzip
  [2] lc0_converted.parquet_v2_suffled.parquet.gzip
  [3] lc0_diverse_converted.parquet
  [4] lc0_diverse_converted_chunk_shuffled.parquet
  [5] lc0_converted.parquet
  [6] train-00000-of-00003.parquet
  [7] lc0_converted_chunk_shuffled.parquet

💾 MODEL NAME:
  Enter a name for your trained model (without extension)
  Example: 'chessmarro_v2', 'mymodel', etc.

🔧 BASE MODEL (Optional):
  [0] None (Train from scratch)
  [1] chessmarro_epoch6.pth
  [2] chessmarro_v6_final.pth
  [3] chessmarro_v3_final.pth
  [4] chessmarro_v8_final.pth
  [5] chessmarro_v4_final.pth
  [6] chessmarro_v9_epoch8.pth
  [7] chessmarro_v9_epoch1.pth
  [8] chessmarro_v9_epoch0.pth
  [9] chessmarro_v9_epoch5.pth
  [10] chessmarro_v5_final.pth
  [11] chessmarro_v9_epoch2.pth
  [12] chessmarro_v9_epoch6.pth
  [13] chessmarro_v9_epoch9.pth
  [14] chessmarro_v3_epoch6.pth
  [15] chessmarro_v9_epoch7.pth
  [16] chessmarro_v9_epo

 4



Enter model name [chessmarro]: 

 chessmarro_v9



Select base model [0]: 

...

Training Epoch 5: 100%|██████████| 2500/2500 [01:10<00:00, 35.67it/s, L_pol=1.49, L_val=0.039] 


🔎 Test Total Loss: 3.5911 | Acc Policy: 37.82% | Value MSE: 0.1021


Training Epoch 6: 100%|██████████| 2500/2500 [01:10<00:00, 35.66it/s, L_pol=1.39, L_val=0.0405]


🔎 Test Total Loss: 3.5984 | Acc Policy: 37.84% | Value MSE: 0.1024
⚠️ Stopping early due to no improvement.
🎯 Entrenamiento finalizado. Mejor precisión de movimiento: 38.71%
💾 Model checkpoint saved: chessmarro_v9_epoch16.pth
🔄 Training iteration 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   board         200000 non-null  object 
 1   best          200000 non-null  int64  
 2   fen_original  200000 non-null  object 
 3   best_uci      200000 non-null  object 
 4   value         200000 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 7.4 GB
200000


Training Epoch 0:  59%|█████▉    | 1487/2500 [00:41<00:28, 35.69it/s, L_pol=2.96, L_val=0.124]